In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS main.silver;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

try:
    # Leitura da tabela da camada Bronze
    df_bronze_customers = spark.table("main.bronze.tb_customers")

    # Mapeamento e Renomeação de Colunas
    df_renomeado = df_bronze_customers \
        .withColumnRenamed("customer_id", "id_consumidor") \
        .withColumnRenamed("customer_unique_id", "id_unico_consumidor") \
        .withColumnRenamed("customer_zip_code_prefix", "prefixo_cep") \
        .withColumnRenamed("customer_name", "nome_consumidor") \
        .withColumnRenamed("customer_city", "cidade") \
        .withColumnRenamed("customer_state", "estado") \
        .withColumnRenamed("customer_gender", "genero") \
        .withColumnRenamed("customer_birth_date", "data_nascimento") \
        .withColumnRenamed("timestamp_ingestion", "data_ingestao") \
        .withColumnRenamed("customer_age", "idade")

    # Transformações: Upper Case para Cidade e Estado
    df_transformado = df_renomeado \
        .withColumn("cidade", F.upper(F.col("cidade"))) \
        .withColumn("estado", F.upper(F.col("estado"))) \
        .withColumn("prefixo_cep", F.col("prefixo_cep").cast("string"))

    # Deduplicação Sênior com Window Functions
    window_spec = Window.partitionBy("id_consumidor").orderBy(F.col("data_ingestao").desc())

    df_deduplicado = df_transformado \
        .withColumn("row_num", F.row_number().over(window_spec)) \
        .filter(F.col("row_num") == 1) \
        .drop("row_num") # Removemos a coluna auxiliar, pois já não é necessária

    # Guardar na camada Silver como tabela Delta (dim_consumidores)
    destino = "main.silver.dim_consumidores"
    df_deduplicado.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"Sucesso: Tabela {destino} criada e deduplicada com sucesso!")

except Exception as e:
    print(f"Erro ao processar dim_consumidores: {e}")

In [0]:
display(spark.table("main.silver.dim_consumidores"))

In [0]:
from pyspark.sql import functions as F

print("Processando a tabela: fat_pedidos")

try:
    # Leitura da tabela da camada Bronze
    df_bronze_orders = spark.table("main.bronze.tb_orders")

    # Tradução dos nomes das colunas originais para o português
    df_renomeado = df_bronze_orders \
        .withColumnRenamed("order_id", "id_pedido") \
        .withColumnRenamed("customer_id", "id_consumidor") \
        .withColumnRenamed("order_status", "status_pedido") \
        .withColumnRenamed("order_purchase_timestamp", "data_compra") \
        .withColumnRenamed("order_approved_at", "data_aprovacao") \
        .withColumnRenamed("order_delivered_carrier_date", "data_envio_transportadora") \
        .withColumnRenamed("order_delivered_customer_date", "data_entrega_cliente") \
        .withColumnRenamed("order_estimated_delivery_date", "data_estimada_entrega") \
        .withColumnRenamed("timestamp_ingestion", "data_ingestao")

    # Mapeamento dos valores da coluna de status
    df_status = df_renomeado.withColumn(
        "status_pedido",
        F.when(F.col("status_pedido") == "delivered", "entregue")
         .when(F.col("status_pedido") == "canceled", "cancelado")
         .when(F.col("status_pedido") == "shipped", "enviado")
         .when(F.col("status_pedido") == "processing", "processando")
         .when(F.col("status_pedido") == "invoiced", "faturado")
         .when(F.col("status_pedido") == "unavailable", "indisponível")
         .when(F.col("status_pedido") == "created", "criado")
         .when(F.col("status_pedido") == "approved", "aprovado")
         .otherwise(F.col("status_pedido"))
    )

    # Criação das novas colunas derivadas de datas
    df_metricas = df_status \
        .withColumn("tempo_entrega_dias", F.datediff(F.col("data_entrega_cliente"), F.col("data_compra"))) \
        .withColumn("tempo_entrega_estimado_dias", F.datediff(F.col("data_estimada_entrega"), F.col("data_compra"))) \
        .withColumn("diferenca_entrega_dias", F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias"))

    # Criação do indicador de entrega no prazo
    # Se o status não for 'entregue', fica 'Não Entregue'.
    # Se a diferença for <= 0 (chegou antes ou no dia exato do prazo), é 'Sim'. Caso contrario, 'Não'.
    df_final = df_metricas.withColumn(
        "entrega_no_prazo",
        F.when(F.col("status_pedido") != "entregue", "Não Entregue")
         .when(F.col("diferenca_entrega_dias") <= 0, "Sim")
         .otherwise("Não")
    )

    # Salvar como tabela Delta
    destino = "main.silver.fat_pedidos"
    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"Sucesso: Tabela {destino} gerada perfeitamente com todas as regras!")

except Exception as e:
    print(f"Erro ao processar fat_pedidos: {e}")

In [0]:
display(spark.table("main.silver.fat_pedidos"))

In [0]:
from pyspark.sql import functions as F

print("Processando a tabela: fat_itens_pedidos")

try:
    # Leitura da tabela da camada Bronze
    df_bronze_order_items = spark.table("main.bronze.tb_order_items")

    # Mapeamento e Renomeação de Colunas
    df_renomeado = df_bronze_order_items \
        .withColumnRenamed("order_id", "id_pedido") \
        .withColumnRenamed("order_item_id", "id_item") \
        .withColumnRenamed("product_id", "id_produto") \
        .withColumnRenamed("seller_id", "id_vendedor") \
        .withColumnRenamed("shipping_limit_date", "data_limite_envio") \
        .withColumnRenamed("price", "preco_BRL") \
        .withColumnRenamed("freight_value", "preco_frete") \
        .withColumnRenamed("timestamp_ingestion", "data_ingestao")

    # Guardar na camada Silver como tabela Delta
    destino = "main.silver.fat_itens_pedidos"
    df_renomeado.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"Sucesso: Tabela {destino} criada com todas as colunas padronizadas!")

except Exception as e:
    print(f"Erro ao processar fat_itens_pedidos: {e}")

In [0]:
from pyspark.sql import functions as F

print("Processando a tabela: fat_pagamentos_pedidos")

try:
    # Leitura da tabela da camada Bronze
    df_bronze_payments = spark.table("main.bronze.tb_order_payments")

    # Renomeação de todas as colunas para português
    df_renomeado = df_bronze_payments \
        .withColumnRenamed("order_id", "id_pedido") \
        .withColumnRenamed("payment_sequential", "sequencial_pagamento") \
        .withColumnRenamed("payment_type", "tipo_pagamento") \
        .withColumnRenamed("payment_installments", "parcelas") \
        .withColumnRenamed("payment_value", "valor_pagamento") \
        .withColumnRenamed("timestamp_ingestion", "data_ingestao")

    # Mapeamento dos tipos de pagamento (Tradução dos valores da coluna)
    df_transformado = df_renomeado.withColumn(
        "tipo_pagamento",
        F.when(F.col("tipo_pagamento") == "credit_card", "Cartão de Crédito")
         .when(F.col("tipo_pagamento") == "boleto", "Boleto")
         .when(F.col("tipo_pagamento") == "voucher", "Voucher")
         .when(F.col("tipo_pagamento") == "debit_card", "Cartão de Débito")
         .when(F.col("tipo_pagamento") == "not_defined", "Não Definido")
         .otherwise(F.col("tipo_pagamento")) # Mantém original caso surja um tipo novo no futuro
    )

    # Guardar na camada Silver como tabela Delta
    destino = "main.silver.fat_pagamentos_pedidos"
    df_transformado.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"Sucesso: Tabela {destino} criada com pagamentos traduzidos!")

except Exception as e:
    print(f"Erro ao processar fat_pagamentos_pedidos: {e}")

In [0]:
fat_pagamentos_pedidos = spark.table("main.silver.fat_pagamentos_pedidos")
fat_pagamentos_pedidos.display()

In [0]:
from pyspark.sql import functions as F

print("Processando a tabela: fat_avaliacoes_pedidos")

try:
    # Leitura da tabela da camada Bronze
    df_bronze_reviews = spark.table("main.bronze.tb_order_reviews")

    # Mapeamento e Renomeação de Colunas
    df_renomeado = df_bronze_reviews \
        .withColumnRenamed("review_id", "id_avaliacao") \
        .withColumnRenamed("order_id", "id_pedido") \
        .withColumnRenamed("review_score", "nota_avaliacao") \
        .withColumnRenamed("review_comment_title", "titulo_avaliacao_comentario") \
        .withColumnRenamed("review_comment_message", "mensagem_avaliacao_comentario") \
        .withColumnRenamed("review_creation_date", "data_criacao_avaliacao") \
        .withColumnRenamed("review_answer_timestamp", "data_resposta_avaliacao") \
        .withColumnRenamed("timestamp_ingestion", "data_ingestao")

    # Tolerância a falhas: Conversão segura de datas
    # F.expr() para invocar o try_to_timestamp diretamente do motor SQL do Databricks
    df_datas_tratadas = df_renomeado \
        .withColumn("data_criacao_avaliacao", F.expr("try_to_timestamp(data_criacao_avaliacao)")) \
        .withColumn("data_resposta_avaliacao", F.expr("try_to_timestamp(data_resposta_avaliacao)"))

    # Tratamento de Nulos
    df_nulos_tratados = df_datas_tratadas.fillna({
        "titulo_avaliacao_comentario": "Sem título",
        "mensagem_avaliacao_comentario": "Sem comentário"
    })

    # Filtragem (Remover pedidos nulos e datas no futuro)
    df_filtrado = df_nulos_tratados \
        .filter(F.col("id_pedido").isNotNull()) \
        .filter(F.col("data_criacao_avaliacao") <= F.current_timestamp())

    # Guardar na camada Silver como tabela Delta
    destino = "main.silver.fat_avaliacoes_pedidos"
    df_filtrado.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"Sucesso: Tabela {destino} processada com limpeza de dados e tolerância a falhas!")

except Exception as e:
    print(f"Erro ao processar fat_avaliacoes_pedidos: {e}")

In [0]:
display(spark.table("main.silver.fat_avaliacoes_pedidos"))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Processando a tabela: dim_produtos")

try:
    # Leitura da tabela da camada Bronze
    df_bronze_products = spark.table("main.bronze.tb_products")

    # Mapeamento e Renomeação de Colunas
    df_renomeado = df_bronze_products \
        .withColumnRenamed("product_id", "id_produto") \
        .withColumnRenamed("product_name", "nome_produto") \
        .withColumnRenamed("product_category_name", "categoria_produto") \
        .withColumnRenamed("product_name_lenght", "tamanho_nome_produto") \
        .withColumnRenamed("product_description_lenght", "tamanho_descricao_produto") \
        .withColumnRenamed("product_photos_qty", "quantidade_fotos") \
        .withColumnRenamed("product_weight_g", "peso_produto_gramas") \
        .withColumnRenamed("product_length_cm", "comprimento_centimetros") \
        .withColumnRenamed("product_height_cm", "altura_centimetros") \
        .withColumnRenamed("product_width_cm", "largura_centimetros") \
        .withColumnRenamed("timestamp_ingestion", "data_ingestao")

    # Deduplicação Sênior com Window Functions
    window_spec = Window.partitionBy("id_produto").orderBy(F.col("data_ingestao").desc())

    df_deduplicado = df_renomeado \
        .withColumn("row_num", F.row_number().over(window_spec)) \
        .filter(F.col("row_num") == 1) \
        .drop("row_num")

    # Guardar na camada Silver como tabela Delta
    destino = "main.silver.dim_produtos"
    df_deduplicado.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"Sucesso: Tabela {destino} criada com o nome do produto mapeado corretamente!")

except Exception as e:
    print(f"Erro ao processar dim_produtos: {e}")

In [0]:
display(spark.table("main.silver.dim_produtos"))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Processando a tabela: dim_vendedores")

try:
    # Leitura da tabela da camada Bronze
    df_bronze_sellers = spark.table("main.bronze.tb_sellers")

    # mapeamento e Renomeação de Colunas
    df_renomeado = df_bronze_sellers \
        .withColumnRenamed("seller_id", "id_vendedor") \
        .withColumnRenamed("seller_name", "nome_vendedor") \
        .withColumnRenamed("seller_zip_code_prefix", "prefixo_cep") \
        .withColumnRenamed("seller_city", "cidade") \
        .withColumnRenamed("seller_state", "estado") \
        .withColumnRenamed("timestamp_ingestion", "data_ingestao")

    # transformações: Upper Case para Cidade e Estado
    df_transformado = df_renomeado \
        .withColumn("cidade", F.upper(F.col("cidade"))) \
        .withColumn("estado", F.upper(F.col("estado"))) \
        .withColumn("prefixo_cep", F.col("prefixo_cep").cast("string"))

    # Deduplicação Sênior com Window Functions
    window_spec = Window.partitionBy("id_vendedor").orderBy(F.col("data_ingestao").desc())

    # Filtramos apenas a versão mais recente do registro do vendedor
    df_deduplicado = df_transformado \
        .withColumn("row_num", F.row_number().over(window_spec)) \
        .filter(F.col("row_num") == 1) \
        .drop("row_num")

    # Guardar na camada Silver como tabela Delta
    destino = "main.silver.dim_vendedores"
    df_deduplicado.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"Sucesso: Tabela {destino} criada, deduplicada e com textos padronizados!")

except Exception as e:
    print(f"Erro ao processar dim_vendedores: {e}")

In [0]:
display(spark.table("main.silver.dim_vendedores")) 

In [0]:
from pyspark.sql import functions as F

print("Processando a tabela: dim_categoria_produtos_traducao")

try:
    # Leitura da tabela da camada Bronze
    df_bronze_translation = spark.table("main.bronze.tb_product_category_name_translation")

    # Mapeamento e Renomeação de Colunas
    df_renomeado = df_bronze_translation \
        .withColumnRenamed("product_category_name", "nome_produto_pt") \
        .withColumnRenamed("product_category_name_english", "nome_produto_en") \
        .withColumnRenamed("timestamp_ingestion", "data_ingestao")

    # Guardar na camada Silver como tabela Delta
    destino = "main.silver.dim_categoria_produtos_traducao"
    df_renomeado.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"Sucesso: Tabela {destino} processada perfeitamente!")

except Exception as e:
    print(f"Erro ao processar dim_categoria_produtos_traducao: {e}")

In [0]:
display(spark.table("main.silver.dim_categoria_produtos_traducao")) 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Processando a tabela: dim_cotacao_dolar")

try:
    # Leitura da tabela da camada Bronze
    df_bronze_dolar = spark.table("main.bronze.tb_cotacao_dolar")

    # Mapeamento e Renomeação inicial
    df_renomeado = df_bronze_dolar \
        .withColumnRenamed("dataHoraCotacao", "data_cotacao") \
        .withColumnRenamed("cotacaoCompra", "valor_dolar") \
        .withColumnRenamed("timestamp_ingestion", "data_ingestao")
    
    # garante que a coluna de data seja do tipo Date (para a sequência funcionar)
    df_renomeado = df_renomeado.withColumn("data_cotacao", F.to_date(F.col("data_cotacao")))

    # 3. Criação do Calendário Contínuo
    # Pegamos a data mínima e máxima que vieram da API
    df_limites = df_renomeado.select(
        F.min("data_cotacao").alias("min_data"),
        F.max("data_cotacao").alias("max_data")
    )
    
    # Geramos uma sequência de dias entre a data mínima e máxima e "explodimos" em linhas
    df_calendario = df_limites.withColumn(
        "data_continua", 
        F.explode(F.sequence(F.col("min_data"), F.col("max_data")))
    ).select("data_continua")

    # Join (União) do Calendário com os Dados Reais
    # O "left" join garante que todos os dias do calendário existam. Os fins de semana ficarão com valor nulo.
    df_join = df_calendario.join(
        df_renomeado,
        df_calendario.data_continua == df_renomeado.data_cotacao,
        "left"
    )

    # Preenchimento de Faltas (Forward Fill) com Window Function
    # Ordenando pela data contínua, olhando desde o início do tempo até a linha atual
    window_ffill = Window.orderBy("data_continua").rowsBetween(Window.unboundedPreceding, Window.currentRow)

    # se sábado for nulo, ele ignora o nulo e pega o último valor válido (sexta)
    df_preenchido = df_join.withColumn(
        "valor_dolar_preenchido",
        F.last("valor_dolar", ignorenulls=True).over(window_ffill)
    )

    # Limpeza e organização final
    df_final = df_preenchido \
        .select(
            F.col("data_continua").alias("data_cotacao"),
            F.col("valor_dolar_preenchido").alias("valor_dolar")
        ) \
        .withColumn("data_ingestao", F.current_timestamp())

    # Guardar na camada Silver como tabela Delta
    destino = "main.silver.dim_cotacao_dolar"
    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"Sucesso: Tabela {destino} gerada com o calendário contínuo e cotações preenchidas!")

except Exception as e:
    print(f"Erro ao processar dim_cotacao_dolar: {e}")

In [0]:
display(spark.table("main.silver.dim_cotacao_dolar"))

In [0]:
from pyspark.sql import functions as F

print("Processando a tabela: fat_pedido_total...")

try:
    # Leitura das tabelas necessárias da camada Silver
    df_pedidos = spark.table("main.silver.fat_pedidos")
    df_pagamentos = spark.table("main.silver.fat_pagamentos_pedidos")
    df_dolar = spark.table("main.silver.dim_cotacao_dolar")

    # Agregação dos Pagamentos
    # Agrupamos por id_pedido e soma tudo para garantir 1 linha por pedido
    df_pagamentos_agrupados = df_pagamentos.groupBy("id_pedido").agg(
        F.sum("valor_pagamento").alias("valor_total_pago_brl")
    )

    # Primeiro Join: Pedidos + Pagamentos Agrupados
    df_pedidos_pagamentos = df_pedidos.join(df_pagamentos_agrupados, "id_pedido", "left")

    # Preparação da Data para o Dólar
    # Convertendo a 'data_compra' (que tem horas) para uma Data simples (YYYY-MM-DD)
    df_pedidos_pagamentos = df_pedidos_pagamentos.withColumn(
        "data_pura_compra", 
        F.to_date(F.col("data_compra"))
    )

    # Segundo Join: Adicionando a cotação do Dólar daquele dia
    df_completo = df_pedidos_pagamentos.join(
        df_dolar,
        df_pedidos_pagamentos.data_pura_compra == df_dolar.data_cotacao,
        "left"
    )

    # Cálculos, Arredondamento e Seleção Final
    # Dividindo o BRL pela cotação e forçando o arredondamento de 2 casas
    df_final = df_completo.withColumn(
        "valor_total_pago_usd", 
        F.col("valor_total_pago_brl") / F.col("valor_dolar")
    ).select(
        F.col("id_pedido"),
        F.col("id_consumidor"),
        F.col("status_pedido").alias("status"), 
        F.round(F.col("valor_total_pago_brl"), 2).alias("valor_total_pago_brl"),
        F.round(F.col("valor_total_pago_usd"), 2).alias("valor_total_pago_usd"),
        F.col("data_compra").alias("data_pedido") 
    )

    # Guardar na camada Silver como tabela Delta
    destino = "main.silver.fat_pedido_total"
    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"Sucesso: Tabela {destino} gerada e consolidada com sucesso!")

except Exception as e:
    print(f"Erro ao processar fat_pedido_total: {e}")

In [0]:
display(spark.table("main.silver.fat_pedido_total")) 

In [0]:
%sql
-- Otimizando a tabela resumo 
OPTIMIZE main.silver.fat_pedido_total 
ZORDER BY (id_pedido, data_pedido);

-- Otimizando a tabela fato original de pedidos (adaptando o nome da data para a coluna que existe nela)
OPTIMIZE main.silver.fat_pedidos 
ZORDER BY (id_pedido, data_compra);